# Notebook 1 — Loading the Svedala Grid

**Course: Cross-Corridor Capacity Analysis with pandapower (Svedala grid)**

Welcome. By the end of this six-notebook series you will have built a working
**cross-corridor capacity analysis tool** for the Svedala transmission grid — the simplified Nordic-style
test model from KTH-EPE.

## Background — where the model comes from

The Svedala model originates as a **CIM CGMES** dataset (the standard ENTSO-E
exchange format for transmission grids). The KTH-EPE notebook
`Pandapower_CIM_import.ipynb`
(<https://github.com/KTH-EPE/CIM_exportimport>) ingests those XML files and
produces a pandapower network. The *simplified* version of that network is
what we use here, exported as five CSV files — one per element table.

## Learning objectives

- Reconstitute a pandapower network from the five CSVs.
- Inspect the four bidding zones (`ZON_NORR`, `ZON_MITT`, `ZON_SYDVÄST`, `ZON_EXTERN`).
- Map the topology with the real geographic coordinates the CIM model carries.

## 1.1  Setup

Make sure you have these CSVs in `data/` next to this notebook:
`buses.csv`, `lines.csv`, `transformers.csv`, `generators.csv`, `loads.csv`.

If you don't, clone <https://github.com/KTH-EPE/CIM_exportimport>, run
`Pandapower_CIM_import.ipynb`, and copy the contents of its `Svedala_csv/`
folder into your `data/` folder.

In [ ]:
import pandapower as pp
import pandas as pd
import matplotlib.pyplot as plt
from svedala_loader import load_svedala     # shared helper

print(f'pandapower version: {pp.__version__}')

## 1.2  Load the network

`load_svedala()` reads the five CSVs into the corresponding pandapower
DataFrames (`net.bus`, `net.line`, `net.trafo`, `net.gen`, `net.load`) and applies
two fix-ups:

1. **Line ratings:** the simplified model has empty `max_i_ka`. The loader
   fills in conservative typical Nordic values per voltage level (1700 MVA at
   400 kV, 570 MVA at 220 kV, 220 MVA at 135 kV).
2. **Slack check:** confirms exactly one generator has `slack=True` (this
   model uses a slack generator instead of an `ext_grid`).

Open `svedala_loader.py` to see the full source.

In [ ]:
net = load_svedala('data')
print(net)

## 1.3  Element tables

Every pandapower network exposes its components as DataFrames. Take a look:

In [ ]:
# Buses - notice the zone column is already populated by the CIM import
net.bus[['name', 'vn_kv', 'zone']].head(10)

In [ ]:
# Lines
net.line[['name', 'from_bus', 'to_bus', 'r_ohm_per_km', 'x_ohm_per_km',
          'max_i_ka', 'in_service']].head(10)

In [ ]:
# Transformers
net.trafo[['name', 'hv_bus', 'lv_bus', 'sn_mva',
           'vn_hv_kv', 'vn_lv_kv', 'in_service']].head(10)

In [ ]:
# Generators - 'type' tells you Hydro vs Thermal, 'slack' marks the slack
net.gen[['name', 'bus', 'type', 'p_mw', 'min_p_mw',
         'max_p_mw', 'slack']].head(15)

In [ ]:
# Loads
net.load[['name', 'bus', 'p_mw', 'q_mvar']].head(10)

## 1.4  Zones — the four bidding zones

The CIM importer carries the **SubGeographicalRegion** of each bus into the
`zone` column. The Svedala model uses four zones that correspond loosely to
the Swedish bidding zones SE1–SE4:

| Zone           | Character                                     |
|----------------|-----------------------------------------------|
| `ZON_NORR`     | North Sweden — hydro-dominated, surplus       |
| `ZON_MITT`     | Central Sweden — mixed generation, large load |
| `ZON_SYDVÄST`  | South-west Sweden — thermal, deficit          |
| `ZON_EXTERN`   | External boundary equivalent (slack lives here) |

In [ ]:
# Bus and generator counts per zone
import pandas as pd
summary = pd.DataFrame({
    'buses_per_zone':       net.bus.zone.value_counts(),
    'gen_pmax_total_mw':    net.gen.groupby(net.bus.loc[net.gen.bus, 'zone'].values).max_p_mw.sum(),
    'load_p_total_mw':      net.load.groupby(net.bus.loc[net.load.bus, 'zone'].values).p_mw.sum(),
})
summary.fillna(0).round(0)

## 1.5  Map the topology

The CIM model carries real geographic coordinates (lat/lon) in `net.bus.geo`
as GeoJSON strings. We extract them and plot the topology over a simple
lat/lon grid — substations colour-coded by zone.

In [ ]:
import json as _json
import numpy as np

def extract_coords(geo_str):
    if pd.isna(geo_str): return (None, None)
    obj = _json.loads(geo_str)
    lon, lat = obj['coordinates']
    return (lon, lat)

coords = net.bus.geo.apply(extract_coords)
net.bus['lon'] = [c[0] for c in coords]
net.bus['lat'] = [c[1] for c in coords]

ZONE_COLOR = {'ZON_NORR':    '#1f77b4',
              'ZON_MITT':    '#2ca02c',
              'ZON_SYDVÄST': '#d62728',
              'ZON_EXTERN':  '#7f7f7f'}

fig, ax = plt.subplots(figsize=(8, 10))
for zone, color in ZONE_COLOR.items():
    sub = net.bus[net.bus.zone == zone]
    ax.scatter(sub.lon, sub.lat, c=color, label=zone, s=40, zorder=3)

# Draw lines
for _, ln in net.line.iterrows():
    fb, tb = ln.from_bus, ln.to_bus
    ax.plot([net.bus.at[fb,'lon'], net.bus.at[tb,'lon']],
            [net.bus.at[fb,'lat'], net.bus.at[tb,'lat']],
            color='gray', linewidth=0.8, zorder=1)

ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Svedala — buses coloured by zone, lines in grey')
ax.legend(loc='lower left', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 1.6  Save the verified network

Save the loaded network to a JSON file so the next notebooks can start from a
single canonical source rather than re-loading from CSV.

In [ ]:
pp.to_json(net, 'data/svedala.json')
print('Saved data/svedala.json')

## 1.7  Exercises

1. How many lines are at 400 kV? At 220 kV? At 135 kV? Use `net.bus.vn_kv` and
   join with `net.line.from_bus`.
2. Which **substation** has the largest installed generation capacity
   (sum of `max_p_mw` across all generators at any bus belonging to it)?
   Hint: substation name comes from `net.bus.zone` is *not* the substation —
   look at the `name` column of the bus, e.g. `'BLOCKET FT51_400.0kV'` and
   strip the trailing `'_<voltage>kV'`.
3. Sum the load per zone (in MW) and compare with the total generation per
   zone. Which zones are *exporters* of power and which are *importers*?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3


---

✅ **Checkpoint reached.** The Svedala grid is loaded, zoned and visualized.

Continue to [Notebook 2 — Base Case Power Flow](02_base_case_powerflow.ipynb).